Good pre-requisites:
- [DL_TP1](./DL_TP1.ipynb) along with its accompanying [Tensors Tutorial](../../tutorials/Tensors.ipynb)
- [3b1b's Deep Learning introduction videos](https://www.youtube.com/watch?v=aircAruvnKk&list=PLZHQObOWTQDNU6R1_67000Dx_ZCJB-3pi): this playlist contains 10 videos, but only the first 4 are relevant to this lab and the MNIST problem

# Exercice 7 : Reconnaissance des chiffres manuscrits avec un réseau MLP sur PyTorch
L’objectif de cet exercice est construire un modèle neuronal capable de reconnaître des chiffres manuscrits $(0-9)$ à l’aide d’un *MLP* (*Multi-Layer Perceptron* ou *Perceptron Multicouche*) sur PyTorch. L'apprentissage de ce modèle repose sur un processus itératif en ajustant ses poids (paramètres) pour minimiser ses erreurs de prédiction. C'est ici qu'intervient la **descente de gradient** (DG) comme une boussole en guidant l'algorithme vers le point (poids + biais) où l'erreur est la plus faible possible (le minimum global). En calculant la pente de la fonction de perte (gradient), elle permet de déterminer dans quelle direction et avec quelle intensité modifier les poids et les biais à chaque itération pour faire converger le modèle vers une précision optimale. Sans DG, le réseau de neurones modifierait ses paramètres au hasard, ce qui rendrait l'apprentissage extrêmement lent, voire impossible. Elle permet de passer d'un simple constat d'échec à une optimisation ciblée.

L’apprentissage d’un MLP repose sur les étapes suivantes :
- Préparation d’un dataset d’images représentatives avec des étiquettes des classes des chiffres.
- **Initialisation** : définition de l'architecture (nombre de couches et de neurones) et on attribue des valeurs aléatoires aux poids (connexions) et aux biais.
- **Propagation avant (Forward Propagation)** : Les données du dataset passent à travers le réseau. Chaque neurone effectue un calcul mathématique pour transmettre l'information jusqu'à la couche de sortie, qui génère une prédiction.
- **Calcul de l'erreur (Loss Function)** : On compare la prédiction du modèle avec la valeur réelle attendue. La différence est mesurée par une "fonction de perte".
- **Rétropropagation (Backpropagation)** : C'est l'étape cruciale. L'erreur est renvoyée en arrière dans le réseau pour identifier quels poids ont le plus contribué à l'imprécision.
- **Optimisation (Descente de gradient)** : On ajuste légèrement les poids et les biais dans la direction qui réduit l'erreur. On répète ensuite les étapes 2 à 5 (pendant plusieurs époques) jusqu'à ce que le modèle soit performant.

Le dataset MNIST est le standard de référence pour développer ce système de reconnaissance de chiffres manuscrits. Contenu : Chiffres (classes) de 0 à 9 écrits à la main par des lycéens et des employés du bureau du recensement américain avec 70 000 images au total qui sont devisées de la façon suivante : 60 000 images (pour ajuster les poids du modèle durant l’entrainement) et 10 000 images (utilisées pour évaluer et tester la capacité de généralisation de ce modèle sur des nouvelles images de chiffres).

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms

1. Télécharger le dataset MNIST en utilisant `torchvision`

`download=True`: the files are downloaded once into the folder `data/`, then reused.

In [5]:
transform = transforms.ToTensor()

# download=True: the files are downloaded once into the folder "data", then reused
train_data = datasets.MNIST(root="data", train=True, download=True)
test_data = datasets.MNIST(root="data", train=False, download=True)

Chaque image est en niveaux de gris de $28 \times 28 = 784$ pixels.

2. Concevoir, entraîner et évaluer un réseau de neurones de type MLP sur PyTorch pour classer ces images dans l'une des 10 classes (0 à 9). Vous devrez analyser aussi l'impact de différents hyper-paramètres sur la performance du modèle. La structure de modèle doit être configurée de la façon suivante :
- Couche d’entrée de 784 neurones (correspondant aux pixels aplatis).
- Couche Cachée de 128 neurones avec une fonction d’activation ReLU.
- Couche de sortie : 10 neurones (un par classe/chiffre) avec une fonction de perte `CrossEntropy`.
- Optimiseur : Adam (tester les autres).

<div align="center">
    <img src="../../assets/MLP.svg"
</div>

The forward propagation equation as follows:

$$ a_i^{(1)} = \Sigma_j p_j w_{i,j} + b_i $$

which is illustrated as:

<div align="center">
    <img src="../../assets/forward.svg"
</div>

In [1]:
class MLP(nn.Module):
    def __init__(self, in_features=784, hidden=128, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_feature, hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.out = nn.Linear(hidden, num_classes)

    
    def forward(self, x):
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.out(x)